# FI-2010 Dataset Exploration

This notebook explores the FI-2010 Limit Order Book dataset for mid-price prediction.

## Dataset Overview

- **Features**: 144 features per timestep
  - 40 price levels (10 bid + 10 ask, normalized)
  - 40 volume levels (10 bid + 10 ask, normalized)
  - 64 derived features
- **Labels**: Mid-price movement direction
  - -1: Downward movement
  - 0: Stationary (no movement)
  - 1: Upward movement
- **Prediction Horizons**: k ∈ {1, 2, 3, 5, 10} time steps ahead

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data.download import FI2010Downloader

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Imports successful")

## 1. Load Data

In [ ]:
# Initialize downloader
downloader = FI2010Downloader()

# Load train and test data
print("Loading training data...")
X_train, y_train = downloader.load_data(split='train')

print("\nLoading test data...")
X_test, y_test = downloader.load_data(split='test')

print("\n" + "="*70)
print("Data loaded successfully!")
print("="*70)

## 2. Data Structure Analysis

In [ ]:
print("Dataset Shapes:")
print("-" * 50)
print(f"Train features: {X_train.shape}")
print(f"Train labels:   {y_train.shape}")
print(f"Test features:  {X_test.shape}")
print(f"Test labels:    {y_test.shape}")
print("-" * 50)
print(f"Total samples:  {X_train.shape[0] + X_test.shape[0]:,}")
print(f"Features:       {X_train.shape[1]}")
print(f"Label horizons: {y_train.shape[1]}")

print("\n" + "="*50)
print("Data Types:")
print("-" * 50)
print(f"Features dtype: {X_train.dtype}")
print(f"Labels dtype:   {y_train.dtype}")

## 3. Feature Distribution Analysis

In [ ]:
# Basic statistics
print("Feature Statistics:")
print("-" * 50)
print(f"Min value:    {X_train.min():.4f}")
print(f"Max value:    {X_train.max():.4f}")
print(f"Mean value:   {X_train.mean():.4f}")
print(f"Std value:    {X_train.std():.4f}")
print(f"Median value: {np.median(X_train):.4f}")

# Check for NaN or Inf
print("\nData Quality:")
print("-" * 50)
print(f"NaN values:      {np.isnan(X_train).sum()}")
print(f"Inf values:      {np.isinf(X_train).sum()}")
print(f"Missing labels:  {np.isnan(y_train).sum()}")

In [ ]:
# Plot feature distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, ax in enumerate(axes):
    feature_idx = i * 20  # Sample features
    ax.hist(X_train[:, feature_idx], bins=50, alpha=0.7, edgecolor='black')
    ax.set_title(f'Feature {feature_idx}', fontsize=10)
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle('Feature Distribution Sample', fontsize=14, y=1.02)
plt.show()

## 4. Label Distribution Analysis

In [ ]:
# Analyze label distribution for each horizon
horizons = ['k=1', 'k=2', 'k=3', 'k=5', 'k=10']
label_names = {-1: 'Down', 0: 'Stationary', 1: 'Up'}

print("Label Distribution by Prediction Horizon:")
print("=" * 70)

for i, horizon in enumerate(horizons):
    print(f"\n{horizon} Label Distribution:")
    print("-" * 50)
    unique, counts = np.unique(y_train[:, i], return_counts=True)
    for label, count in zip(unique, counts):
        percentage = count / len(y_train) * 100
        label_name = label_names.get(int(label), f'Label {int(label)}')
        print(f"  {label_name:12} ({int(label):2d}): {count:7,} ({percentage:5.2f}%)")

print("\n" + "=" * 70)

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for i, (ax, horizon) in enumerate(zip(axes, horizons)):
    unique, counts = np.unique(y_train[:, i], return_counts=True)
    labels = [label_names.get(int(l), f'Label {int(l)}') for l in unique]
    
    colors = ['#e74c3c', '#95a5a6', '#2ecc71']  # Red, Gray, Green
    bars = ax.bar(labels, counts, color=colors, edgecolor='black', linewidth=1.5)
    
    ax.set_title(f'{horizon}', fontsize=12, fontweight='bold')
    ax.set_ylabel('Count' if i == 0 else '')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add percentage labels on bars
    for bar, count in zip(bars, counts):
        height = bar.get_height()
        pct = count / len(y_train) * 100
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{pct:.1f}%', ha='center', va='bottom', fontsize=9)

plt.suptitle('Label Distribution Across Prediction Horizons (Training Set)', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Limit Order Book Visualization

In [ ]:
# Visualize a single LOB snapshot
# First 100 features represent 10 levels × 10 features
sample_idx = 1000
lob_snapshot = X_train[sample_idx, :100].reshape(10, 10)

plt.figure(figsize=(12, 8))
sns.heatmap(lob_snapshot, cmap='RdYlGn', center=0, 
            cbar_kws={'label': 'Normalized Value'},
            linewidths=0.5, linecolor='gray')
plt.title(f'Limit Order Book Snapshot (Sample {sample_idx})', fontsize=14, fontweight='bold')
plt.xlabel('Feature Index', fontsize=12)
plt.ylabel('Depth Level', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Time Series Visualization

In [ ]:
# Plot evolution of first few features over time
window = 1000
fig, axes = plt.subplots(3, 1, figsize=(15, 10))

feature_indices = [0, 10, 20]
for i, (ax, feat_idx) in enumerate(zip(axes, feature_indices)):
    ax.plot(X_train[:window, feat_idx], linewidth=1.5, color=f'C{i}')
    ax.set_title(f'Feature {feat_idx} Time Series', fontsize=12, fontweight='bold')
    ax.set_xlabel('Time Step')
    ax.set_ylabel('Normalized Value')
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)

plt.tight_layout()
plt.suptitle('Feature Time Series Evolution', fontsize=14, fontweight='bold', y=1.00)
plt.show()

## 7. Correlation Analysis

In [ ]:
# Sample features for correlation (all 144 would be too dense)
sample_size = 5000
sample_features = X_train[:sample_size, ::20]  # Every 20th feature
corr_matrix = np.corrcoef(sample_features.T)

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0,
            square=True, linewidths=0.5,
            cbar_kws={'label': 'Correlation Coefficient'})
plt.title('Feature Correlation Matrix (Sampled Features)', fontsize=14, fontweight='bold')
plt.xlabel('Feature Index (every 20th)', fontsize=12)
plt.ylabel('Feature Index (every 20th)', fontsize=12)
plt.tight_layout()
plt.show()

## 8. Key Findings Summary

### Data Quality
- ✓ No missing values (NaN)
- ✓ No infinite values
- ✓ Features are normalized (mean ≈ 0, std ≈ 1)

### Class Distribution
- Slight imbalance across all horizons
- Stationary class (0) is most common (~40%)
- Up and Down classes are roughly balanced (~30% each)
- Class distribution similar across prediction horizons

### Feature Characteristics
- Features follow approximately normal distributions
- Some correlation between adjacent features (expected for LOB data)
- Temporal patterns visible in time series

### Preprocessing Recommendations
1. **Data is already normalized** - no additional normalization needed
2. **Class imbalance** - consider using weighted loss or class weights
3. **Temporal structure** - sliding window approach for sequences
4. **Feature engineering** - LOB data is already preprocessed and feature-engineered

### Next Steps
1. Create PyTorch Dataset with sliding window (Task 3)
2. Implement DeepLOB model architecture (Task 4)
3. Set up training pipeline with MLflow (Task 5)
4. Handle class imbalance during training

In [ ]:
# Save summary statistics
summary = {
    'train_samples': X_train.shape[0],
    'test_samples': X_test.shape[0],
    'n_features': X_train.shape[1],
    'n_horizons': y_train.shape[1],
    'feature_mean': float(X_train.mean()),
    'feature_std': float(X_train.std()),
    'feature_min': float(X_train.min()),
    'feature_max': float(X_train.max()),
}

print("\nDataset Summary:")
print("=" * 50)
for key, value in summary.items():
    if isinstance(value, float):
        print(f"{key:20}: {value:.4f}")
    else:
        print(f"{key:20}: {value:,}")
print("=" * 50)